## Setup

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.status()

In [ ]:
using CairoMakie
using Carlo.ResultTools
using CarloAnalysis
using DataFrames
using FFTW
using HDF5
using JLD2
using LinearAlgebra
using StaticArrays

set_theme!(theme_latexfonts())

In [ ]:
# Boltzmann constant in meV/K
const kB = 8.617333262e-2

In [ ]:
function generate_spins(jobname, task_no)
    fig = Figure(size=(800, 400))

    task_str = lpad(task_no, 4, "0")
    h5open("../jobs/$jobname.data/task$task_str/run0001.dump.h5") do file
        spins = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/spins")
        )
        spin_xs = map(v -> v[1], spins)
        spin_ys = map(v -> v[2], spins)
        spin_zs = map(v -> v[3], spins)
        Lx, Ly = size(spins)
        fig[1,1] = Axis(fig; title="Spins", backgroundcolor="black")
        strength = vec(spin_zs)
        arrows2d!(1:Lx, 1:Ly, spin_xs, spin_ys, lengthscale=0.5, align=:center, color=strength,
                  colorrange=(-1, 1))

        ηs = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/etas")
        )
        η_xs = getindex.(ηs, 1)
        η_ys = getindex.(ηs, 2)
        η_zs = getindex.(ηs, 3)
        Lx, Ly = size(ηs)
        fig[1,2] = Axis(fig; title="ηs", backgroundcolor="black")
        strength = vec(η_zs)
        arrows2d!(1:Lx, 1:Ly, η_xs, η_ys, lengthscale=0.5, align=:center, color=strength,
                  colorrange=(-1, 1))
    end

    return fig
end

In [ ]:
function generate_spinks(jobname, task_no; run_no=1)
    fig = Figure(size=(800, 500))

    task_str = lpad(task_no, 4, "0")
    run_str = lpad(run_no, 4, "0")
    h5open("../jobs/$jobname.data/task$task_str/run$run_str.dump.h5") do file
        rawetas = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/etas")
        )
        etas = zeros(size(rawetas)..., 3)
        for I in eachindex(IndexCartesian(), rawetas)
            etas[I, :] = rawetas[I]
        end
        etaks = fft(etas, (1, 2)) / length(rawetas)
        spin_mags = sum(abs2.(etaks), dims=3)[:,:,1]
        fig[1,1] = ax = Axis(fig; title="ηk correlations")
        hm = heatmap!(ax, spin_mags)
        Colorbar(fig[2,1], hm, vertical=false, flipaxis=false)

        rawspins = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/spins")
        )
        spins = zeros(size(rawspins)..., 3)
        for I in eachindex(IndexCartesian(), rawspins)
            spins[I, :] = rawspins[I]
        end
        spinks = fft(spins, (1, 2)) / length(rawspins)
        spin_mags = sum(abs2.(spinks), dims=3)[:,:,1]
        fig[1,2] = ax = Axis(fig; title="sk correlations")
        hm = heatmap!(ax, spin_mags)
        Colorbar(fig[2,2], hm, vertical=false, flipaxis=false)
    end

    return fig
end

## am = 4

In [ ]:
results = JobResult("../jobs", "am4")

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "stripe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation half M vs. T (stripe init)", xlabel=L"T",
    ylabel=L"C_{M/2}"
)
generate_plot!(ax1, :T, :sk_corr_half_M, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M vs. T (stripe init)", xlabel=L"T",
    ylabel=L"D_M^\parallel"
)
generate_plot!(ax2, :T, :ηk_corr_M, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "afm_afe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation 3K/4 vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"C_{M/2}"
)
generate_plot!(ax1, :T, :sk_corr_part_K, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M' vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"D_M^\parallel"
)
generate_plot!(ax2, :T, :ηk_corr_M2, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "stripe")
ax1 = fig[1,1] = Axis(fig,
    title="Energy vs. T (stripe init)", xlabel=L"T",
    ylabel=L"H"
)
generate_plot!(ax1, :T, :Energy, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="Heat Capacity vs. T (stripe init)", xlabel=L"T",
    ylabel=L"χ"
)
generate_plot!(ax2, :T, :HeatCap, [:er], df; line=true)
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "afm_afe")
ax1 = fig[1,1] = Axis(fig,
    title="Energy vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"H"
)
generate_plot!(ax1, :T, :Energy, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="Heat Capacity vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"χ"
)
generate_plot!(ax2, :T, :HeatCap, [:er], df; line=true)
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
generate_spinks("stripe-anneal-am4", 3+15*6)

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax = Axis(fig, title=L"In-plane $\eta$ M correlation vs. T", xlabel="T", ylabel="ηk")
generate_plot!(ax, :T, [:ηk_corr_M, :ηk_corr_M2, :ηk_corr_M3], [:er], results.data; line=true) do ηk1, ηk2, ηk3
    ηk = ηk1 + ηk2 + ηk3
    real(ηk[1,1] + ηk[2,2])
end
fig[1,2] = ax = Axis(fig, title=L"\text{Sk 3K/4 (C3 invariant) correlation vs. T}", xlabel="T", ylabel="ηk")
generate_plot!(ax, :T, [:sk_corr_part_K, :sk_corr_part_K2, :sk_corr_part_K3], [:er], results.data; line=true) do sk1, sk2, sk3
    sk1 + sk2 + sk3
end
Legend(fig[1,3], ax, merge=true)
fig

In [ ]:
mctimes = get_mctime_data(results, :Energy, :sk_corr_half_M, :sk_corr_part_K3, :ηk_corr_M)
nothing

In [ ]:
CairoMakie.activate!()
i = 3

var1 = :sk_corr_part_K3
var2 = :ηk_corr_M
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
for j in 4:6
    lines!(ax1, real.(mctimes[i + 15j][:, var1]))
    lines!(ax2, [real(ηk[1,1] + ηk[2,2]) for ηk in mctimes[i + 15j][:, var2]])
end
fig

## am = 6

In [ ]:
results = JobResult("../jobs", "am6")

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "stripe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation half M vs. T (stripe init)", xlabel=L"T",
    ylabel=L"C_{M/2}"
)
generate_plot!(ax1, :T, :sk_corr_half_M, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M vs. T (stripe init)", xlabel=L"T",
    ylabel=L"D_M^\parallel"
)
generate_plot!(ax2, :T, :ηk_corr_M, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "afm_afe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation 3K/4 vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"C_{M/2}"
)
generate_plot!(ax1, :T, :sk_corr_part_K, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M' vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"D_M^\parallel"
)
generate_plot!(ax2, :T, :ηk_corr_M2, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
generate_spinks("am6", 3+15*6)

In [ ]:
mctimes = get_mctime_data(results, :Energy, :sk_corr_half_M, :sk_corr_part_K3, :ηk_corr_M)
nothing

In [ ]:
CairoMakie.activate!()
i = 3

var1 = :sk_corr_half_M
var2 = :ηk_corr_M
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
for j in 0:6
    lines!(ax1, real.(mctimes[i + 15j][:, var1]))
    lines!(ax2, [real(ηk[1,1] + ηk[2,2]) for ηk in mctimes[i + 15j][:, var2]])
end
fig